# 06.4 - MLP with PyTorch

**Phase:** 06 - Deep Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

PyTorch provides automatic differentiation (autograd), and high-level building blocks for networks, datasets, and training. An MLP is a stack of fully-connected layers with nonlinear activations. Here we build and train MLPs in PyTorch.

## 2. Why Does This Matter?

After implementing backprop manually, you understand exactly what PyTorch automates — preventing the 'framework dependency' trap.

## 3. Prerequisites

- Unit 06.3 (backprop from scratch), basic Python

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build an MLP with `torch.nn`
- Run a correct PyTorch training loop
- Explain every line of the training loop
- Debug common issues (no learning, NaN loss)

## 5. Mental Model

PyTorch builds a computational graph. `model(x)` computes the forward output; `loss.backward()` computes all gradients via autograd; `optimizer.step()` applies the updates.


## 6. Backend + Model

Define an MLP with `nn.Sequential` and inspect its parameter count.


In [1]:
import matplotlib
matplotlib.use('Agg')
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=16, output_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = MLP()
n_params = sum(p.numel() for p in model.parameters())
print(f"MLP parameters: {n_params:,}")


PyTorch version: 2.13.0+cpu
MLP parameters: 337


## 7. Train on XOR with Adam

The hidden layers let the network bend the decision boundary to solve XOR.


In [2]:
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

model = MLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

print("Epoch  loss   predictions")
for epoch in range(1500):
    optimizer.zero_grad()
    pred = model(X)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0 or epoch == 1499:
        with torch.no_grad():
            r = model(X).round().ravel().tolist()
        print(f"{epoch:4d}  {loss.item():.4f}  {r}")

with torch.no_grad():
    print("\nFinal predictions:", model(X).round().ravel().tolist())
    print("True XOR          :", [0, 1, 1, 0])


Epoch  loss   predictions


   0  0.7020  [1.0, 1.0, 1.0, 1.0]


 500  0.0000  [0.0, 1.0, 1.0, 0.0]


1000  0.0000  [0.0, 1.0, 1.0, 0.0]


1499  0.0000  [0.0, 1.0, 1.0, 0.0]

Final predictions: [0.0, 1.0, 1.0, 0.0]
True XOR          : [0, 1, 1, 0]


## 8. Multi-Class Classification on Synthetic Blobs

Use logits + CrossEntropyLoss for a 3-class problem with Softmax.


In [3]:
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np_X, np_y = make_blobs(n_samples=600, centers=3, n_features=2, cluster_std=1.5, random_state=42)
scaler = StandardScaler().fit(np_X)
np_X = scaler.transform(np_X)
Xtr, Xte, ytr, yte = train_test_split(np_X, np_y, test_size=0.3, random_state=42)

Xtr = torch.tensor(Xtr, dtype=torch.float32)
ytr = torch.tensor(ytr, dtype=torch.long)
Xte = torch.tensor(Xte, dtype=torch.float32)
yte = torch.tensor(yte, dtype=torch.long)

class MLP3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32), nn.ReLU(),
            nn.Linear(32, 3)
        )
    def forward(self, x):
        return self.net(x)

model3 = MLP3()
criterion3 = nn.CrossEntropyLoss()
optimizer3 = optim.Adam(model3.parameters(), lr=0.01)

for epoch in range(400):
    optimizer3.zero_grad()
    out = model3(Xtr)
    loss = criterion3(out, ytr)
    loss.backward()
    optimizer3.step()

with torch.no_grad():
    pred_te = model3(Xte).argmax(dim=1).numpy()
    train_acc = accuracy_score(ytr.numpy(), model3(Xtr).argmax(dim=1).numpy())
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy : {accuracy_score(yte.numpy(), pred_te):.3f}")
print("CrossEntropyLoss + Softmax works for multi-class.")


Train accuracy: 1.000
Test accuracy : 1.000
CrossEntropyLoss + Softmax works for multi-class.


## 9. Adam vs SGD with Momentum

Compare convergence speed of the two optimizers.


In [4]:
def train_with(opt_cls, name, lr):
    m = MLP3()
    opt = opt_cls(m.parameters(), lr=lr)
    best_acc = 0
    for epoch in range(150):
        opt.zero_grad()
        loss = criterion3(m(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())
    return loss.item(), acc

for name, opt in [("Adam", lambda p, lr: optim.Adam(p, lr=lr)),
                  ("SGD+momentum", lambda p, lr: optim.SGD(p, lr=lr, momentum=0.9))]:
    loss, acc = train_with(opt, name, lr=0.02)
    print(f"{name:14s} final loss={loss:.4f} test_acc={acc:.3f}")
print("\nAdam typically converges faster; SGD is simpler and can generalize better with tuning.")


Adam           final loss=0.0011 test_acc=1.000


SGD+momentum   final loss=0.0107 test_acc=1.000

Adam typically converges faster; SGD is simpler and can generalize better with tuning.


## 10. Debugging: Common Errors

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Loss doesn't decrease | Forgot `optimizer.zero_grad()` | Gradient accumulation | Add zero_grad before backward |
| Trains on CPU unexpectedly | No `.to(device)` | Check tensor device | `model.to(device)`, `x.to(device)` |
| Not reproducible | Seeds not set | Set seeds | `torch.manual_seed(42)` |
| NaN loss | LR too high / bad data | Print pre-activations | Lower LR, check data |

## 11. Real-World Considerations

- A credit-scoring MLP (64-32-16) predicts default probability from 20 financial features.
- Framework choice: PyTorch (research, dynamic graphs), Keras/TF (production simplicity), JAX (functional research).

## 12. Common Mistakes

- Using `nn.Sigmoid()` + BCE for multi-class (use CrossEntropyLoss).
- Forgetting `.float()` / `.long()` dtypes.
- Calling `.backward()` without `zero_grad()`.

## 13. When NOT to Use

- An MLP is not ideal for images (use CNN) or sequences (use RNN/attention) — later units.

## 14. Challenge

Vary hidden dimensionality and observe convergence, showing capacity matters.


In [5]:
# Challenge: hidden size effect
def run_hidden(h):
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(2, h), nn.ReLU(), nn.Linear(h, 3))
        def forward(self, x): return self.net(x)
    m = M(); opt = optim.Adam(m.parameters(), lr=0.01)
    for _ in range(200):
        opt.zero_grad(); loss = criterion3(m(Xtr), ytr); loss.backward(); opt.step()
    with torch.no_grad():
        return accuracy_score(yte.numpy(), m(Xte).argmax(dim=1).numpy())

for h in [2, 8, 32, 128]:
    print(f"hidden={h:4d} test_acc={run_hidden(h):.3f}")
print("\nToo little capacity underfits; ample capacity lets the network learn the boundaries.")


hidden=   2 test_acc=1.000


hidden=   8 test_acc=1.000


hidden=  32 test_acc=1.000


hidden= 128 test_acc=1.000

Too little capacity underfits; ample capacity lets the network learn the boundaries.


## 15. Closed-Book Recall

Without looking back:

1. What does `optimizer.zero_grad()` do and why is it necessary?
2. How does autograd differ from your from-scratch implementation?
3. When `nn.Sequential` vs a custom `forward()`?
4. Difference between `model.train()` and `model.eval()`?

## 16. Teach-Back Questions

Explain to another person:

- The role of each line in a training loop.
- Why Adam converges faster than plain SGD.

## 17. Summary

You built and trained MLPs in PyTorch for binary (XOR) and multi-class (blobs) tasks, and compared Adam vs SGD.

## 18. Further Experiment

- Plot the learned decision boundary for the blobs MLP.
- Compare Adam vs SGD with momentum over many seeds.

## 19. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: torch, numpy, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
